# l02 · 2일차 2교시 · KOBIS 영화 목록
**목표**: KOBIS(영화관입장권통합전산망) API로 **지난 주말 박스오피스**를 받아 표로 정리합니다.

| 순서 | 내용 | 새로 배우는 것 |
|---|---|---|
| 1 | KOBIS 문서 읽기 | 문서에서 찾을 네 가지 |
| 2 | 날짜 만들기 | `datetime`, `timedelta`, `strftime` |
| 3 | 요청 ①②③④ | `params`에 키 추가, `faultInfo` 확인 |
| 4 | 표로 정리 | 열 고르기, `rename`, `astype(int)` |
| 5 | 정렬과 거르기 | `df[조건]` |
| 6 | 함수로 묶기 | → **l03_2교시_박스오피스함수.py** |

준비: l01에서 `.env`에 `KOBIS_KEY`를 넣었는지 확인하세요.

> **노트북 사용법** · 셀을 클릭하고 **Shift+Enter**를 누르면 실행되고 다음 셀로 넘어갑니다.
> 셀 왼쪽의 `[1] [2]…`는 **실행한 순서**입니다. 위에서부터 차례로 실행하세요. 중간 셀을 건너뛰면 `NameError`가 납니다.
> 처음부터 깨끗하게 다시 하려면 위쪽 메뉴의 **Restart → Run All**을 누릅니다.

## 1. KOBIS 문서 읽기 · 찾을 네 가지
**① 주소**
`http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchWeeklyBoxOfficeList.json`

**② 파라미터**
| 이름 | 뜻 | 필수 |
|---|---|---|
| `key` | 발급받은 키 | 필수 |
| `targetDt` | 조회 날짜 `YYYYMMDD` (예: `20261018`) | 필수 |
| `weekGb` | `0` 주간(월~일) · `1` **주말(금~일)** · `2` 주중 | 선택 |
| `itemPerPage` | 결과 개수 (기본 10) | 선택 |

**③ 응답 예시 (JSON)**
```
{"boxOfficeResult": {
   "showRange": "20261016~20261018",
   "weeklyBoxOfficeList": [
      {"rank": "1", "movieNm": "영화 A", "openDt": "2026-10-14", "audiCnt": "412345", …}, …]}}
```
**④ 한도와 주의**
- 키마다 하루 요청 수 한도가 있음
- 집계가 끝난 **지난 날짜**로 조회해야 함
- 숫자도 `"412345"`처럼 **글자로** 옴

## 2. 날짜 만들기 · 지난 일요일 구하기
주말 박스오피스는 금~일 집계가 끝나야 나옵니다. 그래서 **이미 끝난 지난 주말의 일요일**을 `targetDt`로 씁니다.

| 쓰는 법 | 하는 일 |
|---|---|
| `date.today()` | 오늘 날짜 |
| `날짜.weekday()` | 요일 번호 · 월 0, 화 1, 수 2 … 일 6 |
| `timedelta(days=n)` | n일. 날짜에 더하거나 빼는 값 |
| `날짜.strftime("%Y%m%d")` | 날짜를 `"20261018"` 모양의 글자로 |

**요일 번호에 1을 더한 날 수만큼 빼면 지난 일요일**입니다.

| 오늘 | weekday() | 빼는 날 수 | 지난 일요일 |
|---|---|---|---|
| 10-19 (월) | 0 | 1 | 10-18 |
| 10-21 (수) | 2 | 3 | 10-18 |
| 10-25 (일) | 6 | 7 | 10-18 |

In [ ]:
from datetime import date, timedelta   # 날짜를 다루는 파이썬 기본 도구

today = date.today()                    # 슬라이드는 date(2026, 10, 21)로 고정해서 설명
yesterday = today - timedelta(days=1)   # 날짜 - 1일
last_sunday = today - timedelta(days=today.weekday() + 1)

print(today, "요일 번호:", today.weekday())
print(yesterday.strftime("%Y%m%d"))
print(last_sunday.strftime("%Y%m%d"))   # targetDt에 넣을 값

> **자주 하는 실수** · `targetDt`에 날짜 객체를 그대로 넣으면 `"2026-10-18"` 모양으로 들어가 문서의 `YYYYMMDD`와 다릅니다. 반드시 `strftime("%Y%m%d")`로 바꿔 넣습니다.

## 3. KOBIS 요청 ①② · params에 키를 더해 보내기
1일차와 달라진 점은 `params`에 `"key"` 한 줄이 늘어난 것뿐입니다.

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()   # .env 읽기

# ① 주소 만들기
BASE = "http://www.kobis.or.kr/kobisopenapi/webservice/rest"
url = BASE + "/boxoffice/searchWeeklyBoxOfficeList.json"
params = {"key": os.getenv("KOBIS_KEY"),                 # .env에서 꺼낸 KOBIS 키
          "targetDt": last_sunday.strftime("%Y%m%d"),    # 지난 일요일
          "weekGb": "1"}                                 # 주말(금~일) 집계

# ② 보내기
res = requests.get(url, params=params, timeout=5)
print(res.status_code)

## 4. KOBIS 요청 ③④ · 확인한 뒤 영화 목록 꺼내기
- ③ `raise_for_status()`로 4xx·5xx를 거르고, **응답 안에 `faultInfo`가 있으면 키 오류**입니다. (키가 틀려도 상태 코드는 200)
- ④ `data → ["boxOfficeResult"] → ["weeklyBoxOfficeList"]` 순서로 한 단계씩 엽니다. 목록은 **딕셔너리가 든 리스트**입니다.

키가 없거나 틀리면 이 셀에서 `키 오류`를 출력한 뒤 `KeyError`가 납니다. 키를 확인하고 다시 실행하세요.

In [ ]:
# ③ 확인하기
res.raise_for_status()                  # 4xx·5xx면 여기서 멈춤
data = res.json()
if "faultInfo" in data:                 # 딕셔너리에 이 키가 있는지 확인
    print("키 오류:", data["faultInfo"]["message"])

# ④ 꺼내기
result = data["boxOfficeResult"]
print(result["showRange"])              # 집계 기간
movies = result["weeklyBoxOfficeList"]  # 영화 목록 (리스트)
print(type(movies), len(movies))
for m in movies:                        # m = 영화 한 편의 정보(딕셔너리)
    print(m["rank"], m["movieNm"])

## 5. 표로 정리 · 열 고르기, 이름 바꾸기, 숫자로
응답에는 열이 18개쯤 있습니다. 필요한 4개만 고르고, 영어 이름을 한글로 바꾸고, 글자로 온 숫자를 진짜 숫자로 바꿉니다.

| 쓰는 법 | 하는 일 |
|---|---|
| `df[["열1", "열2"]]` | 열 여러 개 고르기 · **대괄호 두 겹** (안쪽은 리스트) |
| `df.rename(columns={옛이름: 새이름})` | 열 이름 바꾸기 · 딕셔너리로 |
| `df["열"].astype(int)` | 글자를 정수로 바꾸기 |

**왜 `astype(int)`가 필요할까?** 글자는 첫 글자부터 차례로 비교합니다. `"98765"`와 `"412345"`를 비교하면 첫 글자 9와 4를 비교해 `"98765"`가 더 크다고 판단합니다. 계산·정렬 전에 반드시 숫자로 바꿉니다.

In [ ]:
import pandas as pd

df = pd.DataFrame(movies)               # 딕셔너리 리스트 → 표
print(len(df), "행", len(df.columns), "열")

df = df[["rank", "movieNm", "openDt", "audiCnt"]]   # 필요한 열 4개만
df = df.rename(columns={"rank": "순위", "movieNm": "영화명",
                        "openDt": "개봉일", "audiCnt": "관객수"})

print("글자일 때 최댓값:", df["관객수"].max())   # 글자끼리 비교 → 틀린 답이 나올 수 있음
df["순위"] = df["순위"].astype(int)
df["관객수"] = df["관객수"].astype(int)
print("숫자일 때 최댓값:", df["관객수"].max())   # 진짜 최댓값
df.head(3)

> **자주 하는 실수** · 대괄호를 한 겹만 쓴 `df["rank", "movieNm"]`은 `KeyError`가 납니다. 열 여러 개는 `df[["rank", "movieNm"]]`.

## 6. 정렬과 거르기
**거르기**는 두 번에 나눠 읽습니다.
1. `df["관객수"] >= 100000` → 행마다 `True`·`False`가 든 열
2. `df[True·False 열]` → `True`인 행만 남긴 새 표

개봉일 `"2026-10-14"`는 글자지만 앞에서부터 비교해도 날짜 순서와 같아서 그대로 정렬할 수 있습니다.

In [ ]:
print(df["관객수"] >= 100000)    # 1단계: True/False 열이 어떻게 생겼는지 보기

In [ ]:
big = df[df["관객수"] >= 100000]                  # 거르기: 관객 10만 명 이상
big = big.sort_values("개봉일", ascending=False)   # 정렬: 최근 개봉작부터
big.to_excel("주말_박스오피스.xlsx", index=False)
big

## 7. 함수로 묶기
지금까지의 ①~④와 표 정리를 `get_boxoffice(날짜)` 함수 하나로 묶으면, 날짜만 넣어 한 줄로 영화 표를 받을 수 있습니다.
실패하면 **빈 표**(`pd.DataFrame()`)를 돌려주므로 뒤의 코드가 멈추지 않습니다.

➡️ 함수는 **l03_2교시_박스오피스함수.py**에 있습니다. 실행해 보고 주석을 따라 읽어 보세요.

## 2교시 확인 퀴즈
**Q1.** 무엇이 출력될까요?
```python
today = date(2026, 10, 21)
d = today - timedelta(days=3)
print(d.strftime("%Y%m%d"))
```
**Q2.** 무엇이 출력될까요?
```python
s = pd.Series(["98765", "412345"])
print(s.max())
```
**Q3.** `df[df["관객수"] >= 100000]`이 하는 일은? ① 관객수를 100000으로 바꿈 ② 10만 이상인 행만 남김 ③ 관객수로 정렬함

<details><summary>정답 보기</summary>

Q1 `20261018` · Q2 `98765` (글자 비교라서) · Q3 ②
</details>